# E2-T04 pilot v2: concise responses, 384-token cap

Run these cells in order on an L4 GPU. The first pilot remains intact. This pilot reuses the frozen selection and writes separate v2 artifacts. The Drive copy of `LLM_Bias_Audit_PRD.md` must be updated to the revised local file before the gate renewal. Do not start confirmatory runs until the v2 report and outputs are reviewed.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json
drive.mount("/content/drive")
SHARED = Path("/content/drive/MyDrive/CMPE253-Project").resolve()
assert SHARED.is_dir(), f"Missing project folder: {SHARED}"
REPO = Path("/content/cmpe253-llm-demographic-bias-audit")
print(SHARED)


## 1. Get the committed code and environment


In [ ]:
URL = "https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git"
if (REPO / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["make", "setup-colab"], cwd=REPO, check=True)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=REPO, text=True))


## 2. Renew the data gate for the revised governance lock

This checks the unchanged dataset evidence and the updated PRD. It writes new v2 receipts; it does not overwrite the original approval. If the Drive PRD is stale, update its Drive copy and rerun this cell.


In [ ]:
subprocess.run(["make", "ai-review-gate", f"ROOT={SHARED}"], cwd=REPO, check=True)
gate = SHARED / "manifests/data_gate_approval_v2.json"
assert gate.is_file()
print("Current gate:", gate)


## 3. Verify the original frozen sample


In [ ]:
subprocess.run(["make", "pilot", f"ROOT={SHARED}", "ACTION=plan"], cwd=REPO, check=True)
selection = json.loads((SHARED / "manifests/pilot/selection.json").read_text())
assert selection["cab"]["families"] == 50 and selection["cab"]["prompts"] == 150
assert selection["bbq"]["items"] == 100 and selection["bbq"]["permutations"] == 300
print("Selection verified:", selection["cab"]["families"], "families and", selection["bbq"]["items"], "BBQ items")


## 4. Run M1

Each run is resumable. Reconnect and rerun this cell if Colab disconnects.


In [ ]:
subprocess.run(["make", "pilot", f"ROOT={SHARED}", "ACTION=run-model", "MODEL=M1"], cwd=REPO, check=True)
receipt = json.loads((SHARED / "manifests/pilot_v2/model-M1.json").read_text())
assert receipt["status"] == "complete" and len(receipt["runs"]) == 4
print("M1 completed:", [run["dataset"] + "/" + run["condition"] for run in receipt["runs"]])


## 5. Run M2

Each run is resumable. Reconnect and rerun this cell if Colab disconnects.


In [ ]:
subprocess.run(["make", "pilot", f"ROOT={SHARED}", "ACTION=run-model", "MODEL=M2"], cwd=REPO, check=True)
receipt = json.loads((SHARED / "manifests/pilot_v2/model-M2.json").read_text())
assert receipt["status"] == "complete" and len(receipt["runs"]) == 4
print("M2 completed:", [run["dataset"] + "/" + run["condition"] for run in receipt["runs"]])


## 6. Build and inspect the v2 pilot report

Check all eight cells, zero errors, CAB length finishes, memory, and projected run time. Read the private 20-row output packet before deciding whether the pilot gate passes.


In [ ]:
subprocess.run(["make", "pilot", f"ROOT={SHARED}", "ACTION=report"], cwd=REPO, check=True)
report = json.loads((SHARED / "manifests/pilot_v2/pilot_report.json").read_text())
assert report["checks"]["all_eight_cells_complete"] and report["checks"]["zero_errors"]
for run in report["runs"]:
    print(run["model_id"], run["dataset"], run["condition"], "rows", run["rows"], "length finishes", run.get("length_finish_count", "n/a"), "seconds/row", round(run["seconds_per_row"],2))
print("Projected seconds:", report["projected_full_seconds_by_model"])
print("Review packet:", SHARED / report["review_packet_path"])
